<a href="https://colab.research.google.com/github/niranjan2399/ML-and-PyTorch/blob/main/sentiment_text_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import torch
import torch.nn as nn
from datasets import load_dataset
from torch.utils.data import DataLoader
from collections import Counter

In [ ]:
dataset = load_dataset('imdb')

train_data = dataset['train']
test_data = dataset['test']

print(f"Labels: {set(train_data['label'])}")
print(f"Train size: {len(train_data)}, Test size: {len(test_data)}")

In [8]:
def tokenize(text):
  return text.lower().split()

counter = Counter()
for item in train_data:
  counter.update(tokenize(item['text']))

vocab = {"<unk>": 0}
for word, count in counter.items():
  if count > 5:
    vocab[word] = len(vocab)

print(f"Vocab size: {len(vocab)}")

Vocab size: 40132


In [10]:
def text_to_indices(text):
  return [vocab.get(w, 0) for w in tokenize(text)]

def collate_batch(batch):
  labels, texts, offsets = [], [], [0]

  for item in batch:
    labels.append(item['label'])
    indices = torch.tensor(text_to_indices(item['text']), dtype=torch.long)
    texts.append(indices)
    offsets.append(len(indices))

  labels = torch.tensor(labels, dtype = torch.long)
  texts = torch.cat(texts)
  offsets = torch.tensor(offsets[:-1]).cumsum(dim=0)

  return labels, texts, offsets

In [11]:
train_loader = DataLoader(train_data, batch_size=32, shuffle=True, collate_fn=collate_batch)
test_loader = DataLoader(test_data, batch_size=32, shuffle=True, collate_fn=collate_batch)

In [12]:
class TextClassifier(nn.Module):
  def __init__(self, vocab_size, embed_dim, num_classes):
    super().__init__()

    self.embedding = nn.EmbeddingBag(vocab_size, embed_dim)
    self.net = nn.Linear(embed_dim, num_classes)

  def forward(self, text, offsets):
    embedding = self.embedding(text, offsets)
    return self.net(embedding)


In [15]:
model = TextClassifier(vocab_size=len(vocab), embed_dim=128, num_classes=2)

optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.CrossEntropyLoss()

In [17]:
for epoch in range(10):
  model.train()
  total_loss = 0
  correct = 0
  total = 0

  for labels, text, offsets in train_loader:
    predictions = model(text, offsets)
    loss = loss_fn(predictions, labels)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

    total_loss += loss
    correct += (predictions.argmax(dim=1) == labels).sum().item()
    total += labels.size(0)

  print(f"Epoch {epoch}: Loss={total_loss/total:.4f}, Accuracy={correct/total:.4f}")

Epoch 0: Loss=0.0106, Accuracy=0.8555
Epoch 1: Loss=0.0040, Accuracy=0.9551
Epoch 2: Loss=0.0014, Accuracy=0.9866
Epoch 3: Loss=0.0005, Accuracy=0.9963
Epoch 4: Loss=0.0001, Accuracy=0.9993
Epoch 5: Loss=0.0000, Accuracy=0.9999
Epoch 6: Loss=0.0000, Accuracy=1.0000
Epoch 7: Loss=0.0000, Accuracy=1.0000
Epoch 8: Loss=0.0000, Accuracy=1.0000
Epoch 9: Loss=0.0000, Accuracy=1.0000


In [20]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
  for labels, text, offsets in test_loader:
    predictions = model(text, offsets)
    correct += (predictions.argmax(dim=1) == labels).sum().item()
    total += labels.size(0)

print(f"Test Accuracy: {correct/total:.4f}")

Test Accuracy: 0.8545


In [25]:
def predict_text(text):
    model.eval()

    indices = torch.tensor(text_to_indices(text), dtype=torch.long)
    offsets = torch.tensor([0])

    with torch.no_grad():
        output = model(indices, offsets)
        predicted_class = output.argmax(dim=1).item()

    return "Positive" if predicted_class == 1 else "Negative"

In [38]:
text = "this movie was awesome"
prediction = predict_text(text)

print(f"Prediction:", prediction)

Prediction: Positive
